# 🧬 PROTEUS Studio

### Zero-shot protein conformational-plasticity analysis — interactive, in your browser

Reads the latent trajectory of a flow-matching structure predictor (SimpleFold) to
reveal, with **no training and no MD**: per-residue plasticity, the rigid→IDP
continuum, fold-switch potential, and a 3D plasticity map. Runs on the **free Colab
T4** (ESM-2 3B in fp16). A screening/annotation tool — each readout shows its
validation metric.


## How to start (3 clicks)

1. **Runtime → Change runtime type → T4 GPU**, then **Connect**.
2. **Click ▶ on the single cell below.** First run installs + loads the model (~5 min,
   one time); you'll see **"Ready"**. After that, analyses are fast and **interactive**.
3. The **Studio** opens underneath: add proteins, **Run PROTEUS**, explore. You can
   also upload a **.pdb/.cif** for the 3D view. No coding required.


In [ ]:
#@title 🧬 **PROTEUS Studio** — click the ▶ button to start { display-mode: "form" }
#@markdown First run installs dependencies and loads the model (~5 min, one time). The interactive studio then appears below.
import os, subprocess, sys

PROTEUS_REPO_URL  = "https://github.com/lupiochi/proteus"
SIMPLEFOLD_URL    = "https://github.com/apple/ml-simplefold"
INSTALL_SIMPLEFOLD = True

import shutil

def sh(cmd):
    print("+", cmd)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit("command failed: " + cmd)

# viz stack (always) + transformers (HF ESM-2 backend, fp16/low-RAM)
sh("pip -q install matplotlib plotly anywidget biopython requests numpy transformers")
try:
    from google.colab import output as _cwm
    _cwm.enable_custom_widget_manager()
except Exception:
    pass

# PROTEUS repo: scoring package + extract script + embedding_extraction/ extractor
PROTEUS_DIR = "/content/proteus"
if not os.path.exists(PROTEUS_DIR):
    sh(f"git clone -q {PROTEUS_REPO_URL} {PROTEUS_DIR}")
sys.path.insert(0, PROTEUS_DIR)          # `import proteus`

# The extractor lives in proteus/embedding_extraction/ but its code imports it as a
# top-level package named `embedding`. Alias it so the imports resolve unchanged.
EMB_SRC  = os.path.join(PROTEUS_DIR, "proteus", "embedding_extraction")
EMB_LINK = "/content/embedding"
if os.path.isdir(EMB_SRC) and not os.path.exists(EMB_LINK):
    os.symlink(EMB_SRC, EMB_LINK)
sys.path.insert(0, "/content")           # `import embedding` -> the alias above

# Official SimpleFold (provides inference / model / utils)
SIMPLEFOLD_DIR = "/content/ml-simplefold"
sf_src = os.path.join(SIMPLEFOLD_DIR, "src", "simplefold")
if INSTALL_SIMPLEFOLD:
    if not os.path.exists(SIMPLEFOLD_DIR):
        sh(f"git clone -q {SIMPLEFOLD_URL} {SIMPLEFOLD_DIR}")
        sh(f"pip -q install -e {SIMPLEFOLD_DIR}")
    if os.path.isdir(sf_src):
        sys.path.insert(0, sf_src)       # bare imports: inference, model, utils
    # Overlay the capture-enabled sampler (adds latent capture during sampling;
    # byte-identical to upstream otherwise). Required for PROTEUS extraction.
    patch = os.path.join(PROTEUS_DIR, "simplefold_patch", "sampler.py")
    dest = os.path.join(sf_src, "model", "torch", "sampler.py")
    if os.path.exists(patch) and os.path.isdir(os.path.dirname(dest)):
        shutil.copy(patch, dest)
        print("Applied capture-enabled sampler patch.")
    else:
        print("WARNING: sampler patch not found; extraction will fail without it.")

print("\\nSetup complete.")

import numpy as np, json, warnings
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from IPython.display import HTML, display
warnings.filterwarnings("ignore")

ACCENT = "#5B2C8D"
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
    "axes.edgecolor": "#cccccc", "axes.grid": True, "grid.color": "#eeeeee",
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})


def _show_mpl(fig):
    """Render a matplotlib figure as an inline PNG. This is the reliable path inside
    Colab ipywidgets Output areas — plotly/py3Dmol inject scripts that do NOT execute
    there (leaving blanks), whereas a PNG always displays."""
    display(fig)
    plt.close(fig)

# --- calibrated reference distributions (bundled, self-contained) ---
REFERENCE = json.loads(r"""{"ols":{"intercept":677.1958,"slope":0.416547},"classes":{"denovo":{"n":146,"resid":[-90.44,-73.97,-72.16,-71.63,-69.72,-68.28,-66.95,-63.46,-62.17,-62.09,-61.23,-60.28,-59.04,-58.46,-57.95,-57.28,-57.09,-55.65,-54.89,-52.08,-50.3,-50.06,-49.53,-47.7,-46.03,-43.7,-42.24,-41.58,-41.4,-40.75,-40.23,-38.28,-37.8,-34.79,-32.77,-29.21,-27.83,-25.4,-24.58,-21.6,-20.48,-17.77,-17.52,-15.97,-14.36,-13.94,-13.78,-9.07,-8.64,-6.56,-1.77,-1.48,-1.09,0.24,0.85,3.81,7.07,8.93,8.97,12.79,14.89,16.93,18.72,23.07,23.83,27.5,28.26,32.03,35.17,39.62,41.11,45.4,46.93,51.04,53.96,55.86,62.59,68.21,79.24,124.07],"median":-21.04},"natural":{"n":308,"resid":[-90.94,-82.17,-77.6,-75.05,-72.57,-69.59,-68.66,-66.87,-61.75,-61.11,-59.02,-56.82,-55.19,-51.05,-49.28,-47.45,-44.86,-42.51,-39.2,-38.87,-37.86,-35.76,-35.01,-32.19,-30.01,-28.58,-27.06,-25.65,-24.04,-22.93,-20.55,-19.48,-18.33,-16.27,-15.72,-13.87,-12.35,-11.78,-9.59,-8.49,-7.85,-7.0,-4.49,-2.25,-0.57,0.67,2.54,3.51,6.09,6.2,7.68,9.77,11.01,12.76,13.99,15.45,15.92,17.58,18.37,19.67,20.68,21.7,23.62,24.33,24.87,25.91,27.37,30.26,32.22,34.18,37.7,39.74,40.47,42.26,47.34,48.96,55.67,63.2,76.06,109.66],"median":-8.34},"monostate":{"n":197,"resid":[-152.85,-129.94,-123.25,-121.38,-116.34,-114.0,-112.87,-110.84,-104.25,-101.43,-99.32,-95.64,-94.36,-93.05,-89.43,-88.41,-87.81,-86.61,-82.24,-81.41,-78.7,-76.48,-73.54,-71.65,-70.58,-68.58,-66.07,-65.33,-63.13,-60.12,-59.8,-58.13,-56.52,-52.76,-52.18,-45.93,-44.54,-42.33,-41.18,-35.35,-32.93,-30.69,-29.94,-27.56,-26.52,-24.24,-23.79,-23.3,-21.43,-20.99,-16.94,-16.21,-14.37,-13.21,-11.01,-10.12,-7.08,-6.38,-5.16,-3.67,-2.31,-0.95,5.63,5.78,6.49,9.41,15.31,18.0,21.96,27.94,33.64,37.87,43.64,46.98,83.94,100.59,101.03,101.75,126.92,194.83],"median":-34.61},"foldswitch":{"n":171,"resid":[-128.08,-120.12,-118.11,-113.11,-106.99,-106.22,-104.77,-102.36,-99.31,-85.58,-77.57,-75.99,-74.97,-74.32,-71.51,-70.63,-68.19,-66.74,-64.02,-63.32,-61.55,-57.57,-55.88,-50.51,-47.75,-47.1,-45.19,-42.92,-39.87,-37.72,-34.67,-32.61,-31.34,-29.39,-28.71,-26.64,-24.55,-19.14,-17.69,-16.17,-15.39,-15.37,-12.74,-10.06,-8.86,-8.08,-6.55,-4.62,-3.64,-1.78,1.08,3.9,5.5,8.56,10.34,11.77,17.43,20.96,21.02,25.26,29.93,30.63,31.87,42.26,42.46,48.7,53.61,54.32,60.33,61.82,64.27,76.61,77.92,80.77,83.65,104.35,115.86,135.44,158.34,231.04],"median":-15.79},"dibs":{"n":256,"resid":[-112.91,-94.41,-87.14,-74.72,-67.13,-63.65,-57.97,-54.1,-47.39,-41.76,-40.01,-36.68,-32.76,-28.86,-25.14,-19.16,-15.94,-13.18,-11.58,-10.65,-8.26,-7.58,-7.33,-5.05,0.28,3.49,7.66,12.99,13.59,14.99,17.54,18.4,19.91,23.93,24.49,24.76,25.38,28.73,31.98,35.34,39.13,39.91,40.61,42.58,45.41,47.49,49.0,53.03,55.88,59.29,60.78,69.78,73.75,76.89,79.48,82.56,88.01,90.78,92.93,98.45,105.46,112.54,117.91,122.95,127.37,130.35,133.71,139.26,142.86,152.81,163.81,186.09,200.68,212.14,228.36,234.05,252.36,278.44,315.52,403.64],"median":38.03}},"foldswitch_threshold":-20.07,"foldswitch_auroc":0.77,"note":"resid = l2_delta_max - (intercept + slope*length); reference scores computed with simplefold_360M, K=10 conformations, num_steps=25, tau=0.3. Classes: denovo/natural=Tsuboyama2023, monostate/foldswitch=fold-switch benchmark, dibs=DIBS folding-upon-binding."}""")

CLASS_INFO = [
    ("denovo",     "De novo (rigid)",         "#2c7fb8"),
    ("natural",    "Natural single-domain",   "#41b6c4"),
    ("monostate",  "Monostate",               "#7fcdbb"),
    ("foldswitch", "Fold-switching",          "#fe9929"),
    ("dibs",       "IDP / folds-on-binding",  "#d7301f"),
]
CLASS_NAME = {k: n for k, n, _ in CLASS_INFO}
CLASS_COLOR = {k: c for k, _, c in CLASS_INFO}


def load_npz_embeddings(path):
    """Return (seq_emb [L,D], struct_embs list of [L,D], sequence_str|None).
    Auto-detects key convention: sequence-only = the 2D embedding without a
    _conf/_mean/_var suffix; structural = *_conf* (preferred) or *_mean."""
    d = np.load(path, allow_pickle=True)
    keys = list(d.files)
    conf_keys = sorted(k for k in keys if "_conf" in k)
    mean_keys = [k for k in keys if k.endswith("_mean")]
    struct_set = set(conf_keys) | set(mean_keys) | {k for k in keys if k.endswith("_var")}
    twod = [k for k in keys if getattr(d[k], "ndim", 0) == 2]
    seq_cands = [k for k in twod if k not in struct_set]
    if not seq_cands:
        raise ValueError(f"No sequence-only embedding found among keys: {keys}")
    seq_emb = np.asarray(d[seq_cands[0]], dtype=np.float32)
    if conf_keys:
        struct = [np.asarray(d[k], dtype=np.float32) for k in conf_keys]
    elif mean_keys:
        struct = [np.asarray(d[mean_keys[0]], dtype=np.float32)]
    else:
        raise ValueError(f"No structural conformations found among keys: {keys}")
    seq = str(d["sequence"]) if "sequence" in keys else None
    return seq_emb, struct, seq


def per_residue_profile(seq_emb, struct_embs):
    sm = np.mean(struct_embs, axis=0)
    return np.linalg.norm(sm - seq_emb, axis=-1)


def compute_features(profile):
    return {"l2_delta_max": float(profile.max()),
            "l2_delta_mean": float(profile.mean()),
            "l2_delta_p90": float(np.percentile(profile, 90)),
            "length": int(len(profile))}


def residualize(score, length):
    o = REFERENCE["ols"]
    return score - (o["intercept"] + o["slope"] * length)


def pct_in(resid, cls):
    arr = np.asarray(REFERENCE["classes"][cls]["resid"], float)
    return float((arr < resid).mean() * 100.0)


def assign_continuum(resid):
    diffs = {k: abs(resid - REFERENCE["classes"][k]["median"]) for k, _, _ in CLASS_INFO}
    return min(diffs, key=diffs.get)


def foldswitch_verdict(resid):
    pct = pct_in(resid, "foldswitch")
    thr = REFERENCE["foldswitch_threshold"]
    if pct >= 80:   label = "High"
    elif pct >= 50: label = "Moderate"
    else:           label = "Low"
    return label, pct, bool(resid >= thr)


def flexible_regions(profile, z=1.0, min_len=2):
    thr = profile.mean() + z * profile.std()
    mask = profile > thr
    regions, i, L = [], 0, len(profile)
    while i < L:
        if mask[i]:
            j = i
            while j < L and mask[j]:
                j += 1
            if j - i >= min_len:
                regions.append((i + 1, j))   # 1-indexed inclusive
            i = j
        else:
            i += 1
    return regions, float(thr)


# ----------------------------- plots -----------------------------

def plot_profile(profile, seq=None):
    regs, thr = flexible_regions(profile)
    x = np.arange(1, len(profile) + 1)
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.fill_between(x, profile, color=ACCENT, alpha=0.18)
    ax.plot(x, profile, color=ACCENT, lw=1.8)
    ax.axhline(thr, ls=":", color="#d7301f", lw=1, label="hotspot threshold")
    for a, b in regs:
        ax.axvspan(a - 0.5, b + 0.5, color="#fe9929", alpha=0.18)
    ax.set_xlim(1, len(profile)); ax.set_xlabel("residue")
    ax.set_ylabel("plasticity (L2)")
    ax.set_title("Per-residue PROTEUS plasticity", fontweight="bold",
                 color="#3b1d5e", loc="left")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    fig.tight_layout(); _show_mpl(fig)


def plot_continuum(resid):
    data = [REFERENCE["classes"][k]["resid"] for k, _, _ in CLASS_INFO]
    colors = [c for _, _, c in CLASS_INFO]
    fig, ax = plt.subplots(figsize=(8, 3.6))
    parts = ax.violinplot(data, vert=False, showmeans=True, widths=0.85)
    for i, b in enumerate(parts["bodies"]):
        b.set_facecolor(colors[i]); b.set_alpha(0.6); b.set_edgecolor(colors[i])
    for key in ("cbars", "cmins", "cmaxes", "cmeans"):
        if key in parts:
            parts[key].set_color("#888"); parts[key].set_linewidth(1)
    ax.axvline(resid, color="black", ls="--", lw=2)
    ax.text(resid, len(data) + 0.5, " your protein", fontsize=9, fontweight="bold",
            va="bottom")
    ax.set_yticks(range(1, len(data) + 1))
    ax.set_yticklabels([n for _, n, _ in CLASS_INFO], fontsize=9)
    ax.set_xlabel("length-corrected PROTEUS score")
    ax.set_title("Conformational plasticity continuum", fontweight="bold",
                 color="#3b1d5e", loc="left")
    ax.grid(False)
    fig.tight_layout(); _show_mpl(fig)


def plot_foldswitch(resid):
    label, pct, _ = foldswitch_verdict(resid)
    col = {"Low": "#2c7fb8", "Moderate": "#fe9929", "High": "#d7301f"}[label]
    fig, ax = plt.subplots(figsize=(5, 3.2))
    theta = np.linspace(np.pi, 0, 100)
    ax.plot(np.cos(theta), np.sin(theta), color="#e6e6e6", lw=14, solid_capstyle="round")
    n = max(2, int(pct))
    th2 = np.linspace(np.pi, np.pi - np.pi * n / 100, n)
    ax.plot(np.cos(th2), np.sin(th2), color=col, lw=14, solid_capstyle="round")
    ax.text(0, -0.02, f"{pct:.0f}", ha="center", fontsize=28, fontweight="bold", color=col)
    ax.text(0, -0.30, f"Fold-switch: {label}", ha="center", fontsize=12, color="#3b1d5e")
    ax.text(0, -0.46, "percentile vs known fold-switchers (AUROC 0.77)",
            ha="center", fontsize=8, color="#888")
    ax.set_xlim(-1.25, 1.25); ax.set_ylim(-0.6, 1.18)
    ax.axis("off"); ax.set_aspect("equal")
    fig.tight_layout(); _show_mpl(fig)


def fetch_pdb(seq, uniprot=None):
    import requests
    if uniprot:
        try:
            u = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot.strip()}-F1-model_v4.pdb"
            r = requests.get(u, timeout=30)
            if r.ok and "ATOM" in r.text:
                return r.text, "AlphaFold DB"
        except Exception:
            pass
    if seq and len(seq) <= 400:
        try:
            r = requests.post("https://api.esmatlas.com/foldSequence/v1/pdb/",
                              data=seq, timeout=180, verify=False)
            if r.ok and "ATOM" in r.text:
                return r.text, "ESMFold"
        except Exception as e:
            print("ESMFold fetch failed:", e)
    return None, None


def show_3d(profile, seq=None, uniprot=None):
    """Backbone (Cα) trace coloured by per-residue plasticity. Rendered as a
    matplotlib 3D PNG so it displays inside Colab Output (py3Dmol's WebGL does not)."""
    pdb, src = fetch_pdb(seq, uniprot)
    if pdb is None:
        print("Could not fetch a structure (give a UniProt ID, or use a sequence <= 400 aa).")
        return
    xs, ys, zs, idx = [], [], [], []
    for line in pdb.splitlines():
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            try:
                idx.append(int(line[22:26]))
                xs.append(float(line[30:38])); ys.append(float(line[38:46]))
                zs.append(float(line[46:54]))
            except Exception:
                pass
    if len(xs) < 3:
        print("Could not parse Cα atoms from the structure."); return
    xs, ys, zs = np.array(xs), np.array(ys), np.array(zs)
    col = np.array([profile[i - 1] if 1 <= i <= len(profile) else float(profile.mean())
                    for i in idx])
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(111, projection="3d")
    ax.plot(xs, ys, zs, color="#cccccc", lw=1, zorder=1)
    sc = ax.scatter(xs, ys, zs, c=col, cmap="plasma", s=30, zorder=2)
    fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02, label="PROTEUS plasticity")
    ax.set_title(f"Backbone coloured by plasticity  ({src})", fontweight="bold",
                 color="#3b1d5e")
    ax.set_axis_off()
    fig.tight_layout(); _show_mpl(fig)


def report_card(name, features, resid):
    cls = assign_continuum(resid)
    fs_label, fs_pct, _ = foldswitch_verdict(resid)
    regions, _ = flexible_regions(np.asarray(features["_profile"]))
    reg_str = ", ".join(f"{a}-{b}" for a, b in regions[:6]) or "none above threshold"
    rows = [
        ("Protein", name),
        ("Length", f"{features['length']} aa"),
        ("PROTEUS score (l2_delta_max)", f"{features['l2_delta_max']:.1f}"),
        ("Length-corrected score", f"{resid:+.1f}"),
        ("Nearest continuum class", CLASS_NAME[cls]),
        ("Fold-switch potential", f"{fs_label} ({fs_pct:.0f}th %ile vs fold-switchers)"),
        ("Flexible hotspots (residues)", reg_str),
    ]
    html = ["<table style='border-collapse:collapse;font-family:sans-serif;font-size:14px'>"]
    html.append("<tr><th colspan=2 style='background:#5B2C8D;color:white;padding:8px;"
                "text-align:left'>PROTEUS report</th></tr>")
    for i, (k, val) in enumerate(rows):
        bg = "#f4f0fa" if i % 2 else "#ffffff"
        html.append(f"<tr style='background:{bg}'>"
                    f"<td style='padding:6px 12px;font-weight:600'>{k}</td>"
                    f"<td style='padding:6px 12px'>{val}</td></tr>")
    html.append("</table>")
    display(HTML("".join(html)))

print("PROTEUS library loaded. Reference classes:",
      {k: REFERENCE["classes"][k]["n"] for k, _, _ in CLASS_INFO})

import numpy as np
import matplotlib.pyplot as plt


def per_residue_var(struct_embs):
    """Per-residue conformational ensemble spread (sqrt mean variance across the K
    structural conformations) — the VAR signal from the paper's formulation search."""
    st = np.stack([np.asarray(e, np.float32) for e in struct_embs], axis=0)  # [K,L,D]
    return np.sqrt(st.var(axis=0).mean(axis=-1))                              # [L]


def plot_profile_dual(prof, var=None, seq=None):
    """Stacked tracks: per-residue plasticity (L2) with hotspot shading + VAR."""
    regs, thr = flexible_regions(prof)
    x = np.arange(1, len(prof) + 1)
    rows = 2 if var is not None else 1
    fig, axes = plt.subplots(rows, 1, figsize=(11, 2.5 * rows + 0.8), sharex=True,
                             gridspec_kw=dict(height_ratios=[1.5, 1][:rows], hspace=0.28))
    axes = np.atleast_1d(axes)
    a0 = axes[0]
    a0.fill_between(x, prof, color="#5B2C8D", alpha=0.18)
    a0.plot(x, prof, color="#5B2C8D", lw=1.8)
    a0.axhline(thr, ls=":", color="#d7301f", lw=1, label="hotspot threshold")
    for a, b in regs:
        a0.axvspan(a - 0.5, b + 0.5, color="#fe9929", alpha=0.18)
    a0.set_xlim(1, len(prof)); a0.set_ylabel("plasticity (L2)")
    a0.legend(loc="upper right", fontsize=8, frameon=False)
    a0.set_title("Per-residue plasticity", fontweight="bold", color="#3b1d5e", loc="left")
    if var is not None:
        a1 = axes[1]
        a1.plot(x, var, color="#1b9e9e", lw=1.6)
        a1.fill_between(x, var, color="#1b9e9e", alpha=0.15)
        a1.set_ylabel("ensemble spread")
        a1.set_title("Conformational ensemble spread (VAR)", fontweight="bold",
                     color="#3b1d5e", loc="left")
    axes[-1].set_xlabel("residue")
    fig.tight_layout(); _show_mpl(fig)


def plot_radar(features, resid, prof):
    """Five-axis plasticity fingerprint (all 0-100, calibrated where possible)."""
    _, fs_pct, _ = foldswitch_verdict(resid)
    # nat = pct_in(resid, "natural")
    meds = [REFERENCE["classes"][c]["median"]
            for c in ("denovo", "natural", "monostate", "foldswitch", "dibs")]
    lo, hi = min(meds), max(meds)
    cont = float(np.clip((resid - lo) / (hi - lo + 1e-9) * 100, 0, 100))
    regs, thr = flexible_regions(prof)
    mobile = float((prof > thr).mean() * 100)
    focal = float(min((prof.max() / (prof.mean() + 1e-9) - 1) * 40, 100))
    cats = ["Fold-switch", "Continuum\nposition",
            "Mobile\nfraction", "Focal\nflexibility"]
    vals = [fs_pct, cont, mobile, focal]
    ang = np.linspace(0, 2 * np.pi, len(cats), endpoint=False).tolist()
    fig = plt.figure(figsize=(5, 4.6)); ax = fig.add_subplot(111, projection="polar")
    ax.plot(ang + ang[:1], vals + vals[:1], color="#5B2C8D", lw=2)
    ax.fill(ang + ang[:1], vals + vals[:1], color="#5B2C8D", alpha=0.25)
    ax.set_xticks(ang); ax.set_xticklabels(cats, fontsize=8)
    ax.set_ylim(0, 100); ax.set_yticks([25, 50, 75])
    ax.set_yticklabels(["25", "50", "75"], fontsize=7, color="#999")
    ax.set_title("Plasticity fingerprint", fontweight="bold", color="#3b1d5e", pad=20)
    fig.tight_layout(); _show_mpl(fig)


def plot_continuum_violin(resid):
    """Reuse the engine's matplotlib violin continuum."""
    plot_continuum(resid)


def plot_landscape(rows):
    """Scatter of all samples: length vs length-corrected score, coloured by class."""
    fcolor = {n: c for _, n, c in CLASS_INFO}
    fig, ax = plt.subplots(figsize=(8.5, 4.8))
    for cls in sorted({r["continuum_class"] for r in rows}):
        pts = [r for r in rows if r["continuum_class"] == cls]
        ax.scatter([p["length"] for p in pts], [p["resid"] for p in pts], s=90,
                   color=fcolor.get(cls, "#888"), edgecolor="#333", label=cls, zorder=3)
        for p in pts:
            ax.annotate(p["name"], (p["length"], p["resid"]), fontsize=7,
                        xytext=(0, 7), textcoords="offset points", ha="center")
    for k, name, color in CLASS_INFO:
        ax.axhline(REFERENCE["classes"][k]["median"], ls=":", color=color, alpha=0.5)
        ax.text(0.995, REFERENCE["classes"][k]["median"], name, fontsize=7, color=color,
                va="center", ha="right", transform=ax.get_yaxis_transform())
    ax.set_xlabel("length (aa)"); ax.set_ylabel("length-corrected PROTEUS")
    ax.set_title("Plasticity landscape of your samples", fontweight="bold",
                 color="#3b1d5e", loc="left")
    ax.legend(fontsize=8, frameon=False, loc="best")
    fig.tight_layout(); _show_mpl(fig)


def plot_overlay(items):
    """Overlay length-normalised per-residue profiles of several samples."""
    fig, ax = plt.subplots(figsize=(9, 3.8))
    for name, prof in items:
        prof = np.asarray(prof, float)
        xn = np.linspace(0, 1, len(prof))
        yn = (prof - prof.min()) / (np.ptp(prof) + 1e-9)
        ax.plot(xn, yn, lw=1.6, label=name)
    ax.set_xlabel("relative position (0 = N-term, 1 = C-term)")
    ax.set_ylabel("normalised plasticity")
    ax.set_title("Per-residue plasticity overlay", fontweight="bold",
                 color="#3b1d5e", loc="left")
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout(); _show_mpl(fig)

print("Advanced analyses loaded.")

# ===== in-process inference engine (load once, fast repeat runs) =====
import os, sys, types, torch
sys.path.insert(0, os.path.join(PROTEUS_DIR, "scripts"))
# Expose BOTH the bare modules (inference/model/utils via src/simplefold) AND the
# `simplefold` package itself (via its parent src/), which SimpleFold's config loader
# resolves as `simplefold.configs`.
sys.path.insert(0, os.path.dirname(sf_src))
import os as _os
_os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
_os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
try:
    from transformers.utils import logging as _hflog
    _hflog.set_verbosity_error()
except Exception:
    pass
import extract_embeddings as EE
try:
    import model.torch.sampler as _smod
    _smod.tqdm = lambda _it=None, **_k: (_it if _it is not None else [])
except Exception:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ESM_HF = "facebook/esm2_t36_3B_UR50D"
_M = {}

def load_models():
    if _M:
        return
    dev = torch.device(DEVICE)
    print("Loading SimpleFold 360M + ESM-2 3B (one-time; first run downloads ~6 GB)…")
    args = types.SimpleNamespace(simplefold_model="simplefold_360M",
            ckpt_dir=os.path.expanduser("~/.cache/simplefold"),
            backend="torch", plddt=False, nsample_per_protein=1)
    model, _ = EE.initialize_folding_model(args)
    model.eval()
    em, ed, a2 = EE.load_esm_hf(ESM_HF, dev)
    _M.update(model=model, dev=dev,
              esm_s_combine=model.esm_s_combine.data.clone().to(dev),
              extractor=EE.SimpleFoldEmbeddingExtractor(model=model,
                  config=EE.ExtractionConfig(points=[EE.ExtractionPoint("trunk_out")])),
              flow=EE.LinearPath(),
              sampler=EE.EMSampler(num_timesteps=25, t_start=1e-4, tau=0.3,
                                   log_timesteps=True),
              esm=em, esm_dict=ed, af2=a2)
    print("Models ready — open the PROTEUS Studio below. ↓")

@torch.no_grad()
def extract_seq(seq, n_conf=10):
    load_models()
    dev = _M["dev"]
    esmaa, lengths = EE.encode_sequences_fast([seq], _M["esm_dict"], _M["af2"], dev)
    comb, mask = EE.extract_esm_combined(esmaa, lengths, _M["esm"], _M["esm_s_combine"])
    batch = EE.create_simplefold_batch(comb, [seq], mask, dev)
    emb = EE.extract_perresidue_at_timesteps(
        _M["model"], batch, [0.0, 1.0], _M["extractor"], dev, _M["flow"], _M["sampler"],
        n_conformations=n_conf, save_all_conformations=True, conf_batch_size=1)
    L = len(seq)
    seq_emb = emb["t0.0"][0, :L].float().cpu().numpy()
    confs = [emb[k][0, :L].float().cpu().numpy()
             for k in sorted(emb) if k.startswith("t1.0_conf")]
    prof = per_residue_profile(seq_emb, confs)
    try:
        var = per_residue_var(confs)
    except Exception:
        var = None
    return prof, var, L

# Eager preload at startup so the Studio is ready and runs are fast.
try:
    load_models()
except Exception as _e:
    print("Model preload deferred to first run:", _e)

import os, tempfile
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

# Comparisons use a max-min normalised plasticity score in [0,1] (paper convention);
# the raw l2_delta_max is not surfaced.
_ALLR = [v for c in REFERENCE["classes"] for v in REFERENCE["classes"][c]["resid"]]
_RMIN, _RMAX = float(min(_ALLR)), float(max(_ALLR))


def norm_score(resid):
    """Cross-protein normalisation: where this protein's length-corrected score sits
    in [0,1] across the whole reference set (used for the report card / landscape)."""
    return float(np.clip((resid - _RMIN) / (_RMAX - _RMIN + 1e-9), 0.0, 1.0))


def _norm01(a):
    """Within-protein normalisation of a per-residue array to [0,1] (used for the
    per-residue profile and the 3D map — raw L2 is never compared across proteins)."""
    a = np.asarray(a, float)
    lo = float(a.min())
    return (a - lo) / (float(a.max()) - lo + 1e-9)


def _show_fig(fig):
    """Interactive render inside Colab Output via a FigureWidget (anywidget-backed;
    needs enable_custom_widget_manager(), done in setup)."""
    try:
        display(go.FigureWidget(fig))
    except Exception:
        fig.show()   # fallback


def plot_profile(profile, seq=None):
    plot_profile_dual(profile, None, seq)


def plot_profile_dual(prof, var=None, seq=None):
    prof = np.asarray(prof, float)
    regs, thr = flexible_regions(prof)              # regions/threshold from raw L2
    prof_n = _norm01(prof)                            # display within-protein 0–1
    thr_n = float((thr - prof.min()) / (np.ptp(prof) + 1e-9))
    x = np.arange(1, len(prof) + 1)
    rows = 2 if var is not None else 1
    fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                        row_heights=[0.6, 0.4][:rows],
                        subplot_titles=(["Per-residue plasticity (within-protein, 0–1)"] +
                                        (["Conformational ensemble spread (within-protein, 0–1)"]
                                         if rows == 2 else [])))
    fig.add_trace(go.Scatter(x=x, y=prof_n, fill="tozeroy", name="plasticity",
                  line=dict(color="#5B2C8D", width=2), fillcolor="rgba(91,44,141,0.15)",
                  hovertemplate="res %{x}<br>%{y:.2f}<extra></extra>"), 1, 1)
    fig.add_hline(y=thr_n, line_dash="dot", line_color="#d7301f", row=1, col=1)
    for a, b in regs:
        fig.add_vrect(x0=a - 0.5, x1=b + 0.5, fillcolor="#fe9929", opacity=0.13,
                      line_width=0, row=1, col=1)
    if var is not None:
        v = _norm01(var)
        fig.add_trace(go.Scatter(x=x, y=v, line=dict(color="#1b9e9e", width=2),
                      name="VAR", hovertemplate="res %{x}<br>%{y:.2f}<extra></extra>"),
                      2, 1)
    fig.update_yaxes(range=[0, 1.02], row=1, col=1)
    fig.update_xaxes(title_text="residue", row=rows, col=1)
    fig.update_layout(template="plotly_white", height=180 + 150 * rows,
                      showlegend=False, margin=dict(t=46, b=36))
    _show_fig(fig)


def plot_radar(features, resid, prof):
    prof = np.asarray(prof, float)
    _, fs_pct, _ = foldswitch_verdict(resid)
    # nat = pct_in(resid, "natural")
    meds = [REFERENCE["classes"][c]["median"]
            for c in ("denovo", "natural", "monostate", "foldswitch", "dibs")]
    lo, hi = min(meds), max(meds)
    cont = float(np.clip((resid - lo) / (hi - lo + 1e-9) * 100, 0, 100))
    regs, thr = flexible_regions(prof)
    mobile = float((prof > thr).mean() * 100)
    focal = float(min((prof.max() / (prof.mean() + 1e-9) - 1) * 40, 100))
    cats = ["Fold-switch", "Continuum<br>position",
            "Mobile<br>fraction", "Focal<br>flexibility"]
    vals = [fs_pct, cont, mobile, focal]
    fig = go.Figure(go.Scatterpolar(r=vals + [vals[0]], theta=cats + [cats[0]],
                    fill="toself", line_color="#5B2C8D",
                    fillcolor="rgba(91,44,141,0.25)",
                    hovertemplate="%{theta}: %{r:.0f}<extra></extra>"))
    fig.update_layout(polar=dict(radialaxis=dict(range=[0, 100])), showlegend=False,
                      height=380, title="Plasticity fingerprint", margin=dict(t=60, b=30))
    _show_fig(fig)


def plot_continuum(resid):
    fig = go.Figure()
    for k, name, color in CLASS_INFO:
        fig.add_trace(go.Violin(x=REFERENCE["classes"][k]["resid"], name=name,
                      line_color=color, fillcolor=color, opacity=0.6, orientation="h",
                      points=False, meanline_visible=True, width=1.7))
    fig.add_vline(x=resid, line_dash="dash", line_color="black", line_width=2,
                  annotation_text="your protein")
    fig.update_layout(template="plotly_white", height=420, showlegend=False,
                      title="Conformational plasticity continuum (length-corrected)",
                      xaxis_title="length-corrected PROTEUS score", margin=dict(t=46, b=36))
    _show_fig(fig)


def plot_continuum_violin(resid):
    plot_continuum(resid)


def plot_foldswitch(resid):
    lab, pct, _ = foldswitch_verdict(resid)
    col = {"Low": "#2c7fb8", "Moderate": "#fe9929", "High": "#d7301f"}[lab]
    fig = go.Figure(go.Indicator(mode="gauge+number", value=pct,
                    number={"suffix": " %ile"},
                    title={"text": f"Fold-switch potential: <b>{lab}</b><br>"
                           "<span style='font-size:.8em'>percentile vs known "
                           "fold-switchers (AUROC 0.77)</span>"},
                    gauge={"axis": {"range": [0, 100]}, "bar": {"color": col},
                           "steps": [{"range": [0, 50], "color": "#eef6fb"},
                                     {"range": [50, 80], "color": "#fff3df"},
                                     {"range": [80, 100], "color": "#fde6e0"}]}))
    fig.update_layout(height=320, margin=dict(t=80, b=10))
    _show_fig(fig)


def plot_landscape(rows):
    fcolor = {n: c for _, n, c in CLASS_INFO}
    fig = go.Figure()
    for cls in sorted({r["continuum_class"] for r in rows}):
        pts = [r for r in rows if r["continuum_class"] == cls]
        fig.add_trace(go.Scatter(x=[p["length"] for p in pts],
                      y=[norm_score(p["resid"]) for p in pts],
                      mode="markers+text", text=[p["name"] for p in pts],
                      textposition="top center", name=cls,
                      marker=dict(size=13, color=fcolor.get(cls, "#888"),
                                  line=dict(width=1, color="#333")),
                      hovertemplate="%{text}<br>len %{x}<br>plasticity %{y:.2f}<extra></extra>"))
    for k, name, color in CLASS_INFO:
        fig.add_hline(y=norm_score(REFERENCE["classes"][k]["median"]), line_dash="dot",
                      line_color=color, opacity=0.5, annotation_text=name,
                      annotation_position="right", annotation_font_size=10)
    fig.update_layout(template="plotly_white", height=470,
                      title="Plasticity landscape of your samples",
                      xaxis_title="length (aa)",
                      yaxis_title="normalised plasticity (0–1)", margin=dict(t=50, b=40))
    _show_fig(fig)


def plot_overlay(items):
    fig = go.Figure()
    for name, prof in items:
        prof = np.asarray(prof, float)
        xn = np.linspace(0, 1, len(prof))
        yn = (prof - prof.min()) / (np.ptp(prof) + 1e-9)
        fig.add_trace(go.Scatter(x=xn, y=yn, mode="lines", name=name,
                      hovertemplate=name + "<br>%{x:.2f}<br>%{y:.2f}<extra></extra>"))
    fig.update_layout(template="plotly_white", height=430,
                      title="Normalised per-residue plasticity (overlay)",
                      xaxis_title="relative position (0=N, 1=C)",
                      yaxis_title="normalised plasticity", margin=dict(t=50, b=40))
    _show_fig(fig)


def _parse_ca(path):
    """Cα residue indices + coords from a .pdb or .cif file (Biopython)."""
    from Bio.PDB import PDBParser, MMCIFParser
    p = MMCIFParser(QUIET=True) if path.lower().endswith(".cif") else PDBParser(QUIET=True)
    s = p.get_structure("s", path)
    chain = next(iter(next(iter(s))))
    idx, xyz = [], []
    for res in chain:
        if "CA" in res:
            idx.append(res.id[1]); xyz.append([float(c) for c in res["CA"].coord])
    return idx, np.array(xyz, float)


def show_3d(profile, seq=None, uniprot=None, struct_path=None):
    """Interactive Cα backbone coloured by plasticity (plotly Scatter3d). Uses an
    uploaded .pdb/.cif if provided, else fetches AlphaFold/ESMFold."""
    profile = np.asarray(profile, float)
    src = None
    if struct_path and os.path.exists(struct_path):
        try:
            idx, xyz = _parse_ca(struct_path); src = "uploaded " + os.path.basename(struct_path)
        except Exception as e:
            print("Could not parse uploaded structure:", e); return
    else:
        pdb, src = fetch_pdb(seq, uniprot)
        if pdb is None:
            print("No structure: upload a .pdb/.cif above, or use a sequence ≤ 400 aa.")
            return
        with tempfile.NamedTemporaryFile("w", suffix=".pdb", delete=False) as fh:
            fh.write(pdb); tmp = fh.name
        idx, xyz = _parse_ca(tmp)
    if len(xyz) < 3:
        print("Could not parse Cα atoms from the structure."); return
    pn = _norm01(profile)                            # within-protein 0–1, fixed colour scale
    col = [float(pn[i - 1]) if 1 <= i <= len(pn) else float(pn.mean())
           for i in idx]
    fig = go.Figure(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2], mode="lines+markers",
        line=dict(color="#cccccc", width=3),
        marker=dict(size=4, color=col, colorscale="Plasma", cmin=0.0, cmax=1.0,
                    colorbar=dict(title="plasticity<br>(0–1)")),
        hovertemplate="res %{text}<extra></extra>", text=[str(i) for i in idx]))
    fig.update_layout(title=f"Backbone coloured by plasticity ({src})", height=560,
                      scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                                 zaxis=dict(visible=False)), margin=dict(t=40, b=0, l=0, r=0))
    _show_fig(fig)


def report_card(name, features, resid):
    cls = assign_continuum(resid); fs_label, fs_pct, _ = foldswitch_verdict(resid)
    # nat = pct_in(resid, "natural")
    regs, _ = flexible_regions(np.asarray(features["_profile"]))
    reg_str = ", ".join(f"{a}-{b}" for a, b in regs[:6]) or "none above threshold"
    rows = [("Protein", name), ("Length", f"{features['length']} aa"),
            ("Plasticity score (0–1, normalised)", f"{norm_score(resid):.2f}"),
            ("Nearest continuum class", CLASS_NAME[cls]),
            ("Fold-switch potential", f"{fs_label} ({fs_pct:.0f}th %ile vs fold-switchers)"),
            ("Flexible hotspots (residues)", reg_str)]
    h = ["<table style='border-collapse:collapse;font-family:sans-serif;font-size:14px'>",
         "<tr><th colspan=2 style='background:#5B2C8D;color:#fff;padding:8px;"
         "text-align:left'>PROTEUS report</th></tr>"]
    for i, (k, val) in enumerate(rows):
        bg = "#f4f0fa" if i % 2 else "#ffffff"
        h.append(f"<tr style='background:{bg}'><td style='padding:6px 12px;font-weight:600;"
                 f"color:#1a1a1a'>{k}</td><td style='padding:6px 12px;color:#1a1a1a'>"
                 f"{val}</td></tr>")
    h.append("</table>")
    display(HTML("".join(h)))

print("Interactive plots loaded.")

import os, re, glob, json, io, csv, subprocess, time
import numpy as np
import ipywidgets as W
from IPython.display import display, HTML, clear_output

ACCENT = "#5B2C8D"; ACCENT2 = "#8e44c9"; OKC = "#2e8b57"; ERRC = "#c0392b"

display(HTML("""
<style>
  /* Force a light, theme-independent surface so text is readable in Colab dark mode */
  .proteus-studio{background:#ffffff;color:#1a1a1a;border-radius:14px;
                  padding:14px 16px;border:1px solid #e6e0f0;}
  .proteus-studio, .proteus-studio *{color:#1a1a1a;}
  .proteus-studio .pst-card{border:1px solid #e6e0f0;border-radius:12px;
            padding:14px 16px;margin:8px 0;background:#ffffff;
            box-shadow:0 1px 3px rgba(0,0,0,.06);}
  .proteus-studio .pst-h{font-family:'Helvetica Neue',sans-serif;font-weight:700;
            color:#3b1d5e!important;font-size:15px;margin:0 0 6px 0;}
  .proteus-studio .pst-sub{color:#6a5a82!important;font-size:12px;margin:0 0 8px 0;}
  .proteus-studio .pst-badge{display:inline-block;padding:2px 9px;border-radius:10px;
            font-size:11px;font-weight:600;color:#fff!important;}
  .proteus-studio table{border-collapse:collapse;font-family:sans-serif;font-size:13px;
            width:100%;background:#ffffff;}
  .proteus-studio table th{background:#5B2C8D;color:#ffffff!important;padding:7px 10px;
            text-align:left;}
  .proteus-studio table td{padding:6px 10px;border-bottom:1px solid #eee;color:#1a1a1a;}
  .proteus-studio table tr:nth-child(even){background:#f7f3fb;}
  .proteus-studio .widget-button{font-weight:600!important;}
  .proteus-banner{background:linear-gradient(90deg,#5B2C8D,#8e44c9);color:#fff!important;
            padding:14px 18px;border-radius:12px;margin-bottom:6px;}
  .proteus-banner *{color:#fff!important;}
</style>"""))

# ---------------- state ----------------
SAMPLES = {}                       # name -> dict(seq,length,status,npz,profile,features,resid)
WORK = "/content/work"; EMB = os.path.join(WORK, "emb")
os.makedirs(EMB, exist_ok=True)

import torch
CUDA = torch.cuda.is_available()
DEVICE = "cuda" if CUDA else "cpu"

def _sanitize(name, i):
    n = re.sub(r"[^A-Za-z0-9_.-]", "_", (name or "").strip())
    return n or f"seq{i}"

def _parse_fasta(text):
    """Return list of (name, seq) from FASTA text or a bare sequence."""
    text = text.strip()
    if not text:
        return []
    out, name, buf = [], None, []
    if not text.startswith(">"):
        # bare sequence(s), one per non-empty line OR a single block
        seq = "".join(text.split())
        return [(None, seq.upper())] if seq else []
    for line in text.splitlines():
        line = line.strip()
        if line.startswith(">"):
            if name is not None and buf:
                out.append((name, "".join(buf).upper()))
            name = line[1:].split()[0] if len(line) > 1 else None
            buf = []
        elif line:
            buf.append(line)
    if name is not None and buf:
        out.append((name, "".join(buf).upper()))
    return out

VALID_AA = set("ACDEFGHIKLMNPQRSTVWYX")
def _clean_seq(s):
    return "".join(c for c in s.upper() if c in VALID_AA)

# ---------------- widgets: ADD ----------------
seq_box = W.Textarea(placeholder=">my_protein\nMKTAYIAK...\n\n(paste FASTA, or one bare sequence)",
                     layout=W.Layout(width="100%", height="120px"))
add_btn = W.Button(description="➕ Add to queue", button_style="info",
                   layout=W.Layout(width="160px"))
clear_btn = W.Button(description="🗑 Clear queue", layout=W.Layout(width="140px"))
upload = W.FileUpload(accept=".fasta,.fa,.txt,.npz", multiple=True,
                      description="⬆ Upload", layout=W.Layout(width="160px"))
queue_out = W.Output()

# ---------------- widgets: RUN ----------------
nconf = W.Dropdown(options=[("5 (fast)", 5), ("10 (default)", 10), ("20 (smoother)", 20)],
                   value=10, description="Conformations:",
                   style={"description_width": "initial"}, layout=W.Layout(width="240px"))
run_btn = W.Button(description="🚀 Run PROTEUS on queue", button_style="success",
                   layout=W.Layout(width="240px"))
run_out = W.Output()

# ---------------- widgets: RESULTS ----------------
result_sel = W.Dropdown(options=[], description="Sample:",
                        style={"description_width": "initial"},
                        layout=W.Layout(width="380px"))
show3d = W.Checkbox(value=True, description="3D structure", indent=False,
                    layout=W.Layout(width="140px"))
show_btn = W.Button(description="🔬 Show analysis", button_style="primary",
                    layout=W.Layout(width="180px"))
result_out = W.Output()
struct_up = W.FileUpload(accept=".pdb,.cif", multiple=False,
                         description="⬆ .pdb/.cif", layout=W.Layout(width="200px"))

# ---------------- widgets: COMPARE ----------------
summary_btn = W.Button(description="📊 Comparison table", layout=W.Layout(width="190px"))
landscape_btn = W.Button(description="🗺 Plasticity landscape", layout=W.Layout(width="210px"))
overlay_sel = W.SelectMultiple(options=[], description="Overlay:",
                               style={"description_width": "initial"},
                               layout=W.Layout(width="380px", height="110px"))
overlay_btn = W.Button(description="📈 Overlay profiles", layout=W.Layout(width="190px"))
dl_csv = W.Button(description="⬇ CSV", layout=W.Layout(width="110px"))
dl_json = W.Button(description="⬇ JSON", layout=W.Layout(width="110px"))
summary_out = W.Output()


def _status_badge(st):
    col = {"queued": "#b08900", "done": OKC, "error": ERRC, "running": ACCENT2}.get(st, "#888")
    return f"<span class='pst-badge' style='background:{col}'>{st}</span>"

def render_queue():
    with queue_out:
        clear_output()
        if not SAMPLES:
            display(HTML("<div class='pst-sub'>Queue is empty — add proteins above.</div>"))
            return
        rows = ["<table class='pst'><tr><th>#</th><th>Name</th><th>Length</th>"
                "<th>Status</th><th>PROTEUS</th><th>Fold-switch</th></tr>"]
        for i, (nm, s) in enumerate(SAMPLES.items(), 1):
            sc = f"{s['features']['l2_delta_max']:.0f}" if s.get("features") else "—"
            fs = "—"
            if s.get("resid") is not None:
                lab, pct, _ = foldswitch_verdict(s["resid"])
                fs = f"{lab} ({pct:.0f}%)"
            rows.append(f"<tr><td>{i}</td><td><b>{nm}</b></td><td>{s['length']}</td>"
                        f"<td>{_status_badge(s['status'])}</td><td>{sc}</td><td>{fs}</td></tr>")
        rows.append("</table>")
        display(HTML("".join(rows)))

def _add_sample(name, seq, i):
    seq = _clean_seq(seq)
    if len(seq) < 10:
        return
    nm = _sanitize(name, i)
    base, k = nm, 2
    while nm in SAMPLES:
        nm = f"{base}_{k}"; k += 1
    SAMPLES[nm] = {"seq": seq, "length": len(seq), "status": "queued",
                   "npz": None, "profile": None, "features": None, "resid": None}

def on_add(_):
    for j, (nm, sq) in enumerate(_parse_fasta(seq_box.value), 1):
        _add_sample(nm or f"protein_{len(SAMPLES)+1}", sq, len(SAMPLES) + 1)
    seq_box.value = ""
    render_queue(); refresh_results_dropdown()

def on_clear(_):
    SAMPLES.clear(); render_queue(); refresh_results_dropdown()

def on_upload(change):
    for fname, meta in upload.value.items() if isinstance(upload.value, dict) else \
            [(f["name"], f) for f in upload.value]:
        content = meta["content"] if isinstance(meta, dict) else meta
        if fname.lower().endswith(".npz"):
            # pre-computed embedding: register as already-done
            path = os.path.join(EMB, "embeddings"); os.makedirs(path, exist_ok=True)
            nm = _sanitize(os.path.splitext(fname)[0], len(SAMPLES) + 1)
            with open(os.path.join(path, nm + ".npz"), "wb") as fh:
                fh.write(bytes(content))
            try:
                _load_one(nm, os.path.join(path, nm + ".npz"))
            except Exception as e:
                print("npz load failed:", e)
        else:
            txt = bytes(content).decode("utf-8", "ignore")
            for nm, sq in _parse_fasta(txt):
                _add_sample(nm or f"protein_{len(SAMPLES)+1}", sq, len(SAMPLES) + 1)
    render_queue(); refresh_results_dropdown()

add_btn.on_click(on_add); clear_btn.on_click(on_clear); upload.observe(on_upload, names="value")


def _load_one(nm, npz_path):
    seq_emb, struct, npz_seq = load_npz_embeddings(npz_path)
    prof = per_residue_profile(seq_emb, struct)
    try:
        var = per_residue_var(struct)
    except Exception:
        var = None
    feats = compute_features(prof); feats["_profile"] = prof.tolist()
    resid = residualize(feats["l2_delta_max"], feats["length"])
    if nm not in SAMPLES:
        SAMPLES[nm] = {"seq": npz_seq or "", "length": len(prof)}
    SAMPLES[nm].update({"status": "done", "npz": npz_path, "profile": prof,
                        "var": var, "features": feats, "resid": resid})


def on_run(_):
    with run_out:
        clear_output()
        todo = [nm for nm, s in SAMPLES.items() if s["status"] in ("queued", "error")]
        if not todo:
            print("Nothing to run — add proteins first (or all are done)."); return
        if not CUDA:
            print("WARNING: no GPU detected — this will be slow. "
                  "Runtime > Change runtime type > T4 GPU.")
        if not _M:
            print("Loading models (one-time)…")
        from tqdm.auto import tqdm as _tqdm
        ok = 0; t0 = time.time()
        for nm in _tqdm(todo, desc="PROTEUS"):
            SAMPLES[nm]["status"] = "running"; render_queue()
            try:
                prof, var, _ = extract_seq(SAMPLES[nm]["seq"], nconf.value)
                feats = compute_features(prof); feats["_profile"] = prof.tolist()
                resid = residualize(feats["l2_delta_max"], feats["length"])
                SAMPLES[nm].update(status="done", profile=prof, var=var,
                                   features=feats, resid=resid)
                print(f"   ✓ PROTEUS score for {nm} done.")
                ok += 1
            except Exception:
                SAMPLES[nm]["status"] = "error"
                import traceback
                print("   ✗ error:"); traceback.print_exc()
            render_queue()
        print(f"\nDone in {time.time()-t0:.0f}s — {ok}/{len(todo)} succeeded.")
        refresh_results_dropdown()
        if result_sel.options:                       # auto-show the first result
            result_sel.value = result_sel.options[0]
            on_show(None)

run_btn.on_click(on_run)


def refresh_results_dropdown():
    done = [nm for nm, s in SAMPLES.items() if s["status"] == "done"]
    result_sel.options = done
    overlay_sel.options = done
    if done and result_sel.value not in done:
        result_sel.value = done[0]

def on_show(_):
    with result_out:
        clear_output()
        nm = result_sel.value
        if not nm or SAMPLES.get(nm, {}).get("status") != "done":
            display(HTML("<div class='pst-sub'>Run the queue, then pick a sample.</div>")); return
        s = SAMPLES[nm]; prof = np.asarray(s["profile"])
        var = np.asarray(s["var"]) if s.get("var") is not None else None
        display(HTML(f"<div class='pst-h'>🧬 {nm} &nbsp;<span class='pst-sub'>"
                     f"{s['length']} aa</span></div>"))
        report_card(nm, s["features"], s["resid"])
        plot_profile_dual(prof, var, s.get("seq"))
        display(W.HBox([_polar_out(s, prof), _fs_out(s)]))
        plot_continuum_violin(s["resid"])
        if show3d.value:
            display(HTML("<div class='pst-sub'>Fetching structure for 3D view…</div>"))
            try:
                show_3d(prof, seq=s.get("seq"), struct_path=s.get("struct_path"))
            except Exception as e:
                print("3D view unavailable:", e)


def _polar_out(s, prof):
    o = W.Output(layout=W.Layout(width="49%"))
    with o:
        plot_radar(s["features"], s["resid"], prof)
    return o

def _fs_out(s):
    o = W.Output(layout=W.Layout(width="49%"))
    with o:
        plot_foldswitch(s["resid"])
    return o

show_btn.on_click(on_show)

def on_struct_upload(change):
    nm = result_sel.value
    if not nm:
        return
    items = (struct_up.value.items() if isinstance(struct_up.value, dict)
             else [(f["name"], f) for f in struct_up.value])
    for fname, meta in items:
        content = meta["content"] if isinstance(meta, dict) else meta
        os.makedirs("/content/structures", exist_ok=True)
        ext = ".cif" if fname.lower().endswith(".cif") else ".pdb"
        path = f"/content/structures/{nm}{ext}"
        with open(path, "wb") as fh:
            fh.write(bytes(content))
        SAMPLES[nm]["struct_path"] = path
    on_show(None)
struct_up.observe(on_struct_upload, names="value")
result_sel.observe(lambda c: on_show(None), names="value")


def _summary_rows():
    rows = []
    for nm, s in SAMPLES.items():
        if s["status"] != "done":
            continue
        lab, pct, _ = foldswitch_verdict(s["resid"])
        cls = CLASS_NAME[assign_continuum(s["resid"])]
        regs, _ = flexible_regions(np.asarray(s["profile"]))
        rows.append({"name": nm, "length": s["length"],
                     "plasticity": round(norm_score(s["resid"]), 2),
                     "length_corrected": round(s["resid"], 1),
                     "continuum_class": cls,
                     "foldswitch": lab, "foldswitch_pct": round(pct, 1),
                     "n_hotspots": len(regs)})
    rows.sort(key=lambda r: r["length_corrected"], reverse=True)
    return rows

def on_summary(_):
    with summary_out:
        clear_output()
        rows = _summary_rows()
        if not rows:
            display(HTML("<div class='pst-sub'>No completed samples yet.</div>")); return
        h = ["<table class='pst'><tr><th>Protein</th><th>Len</th>"
             "<th>Plasticity (0–1)</th><th>Continuum class</th><th>Fold-switch</th>"
             "<th>Hotspots</th></tr>"]
        for r in rows:
            h.append(f"<tr><td><b>{r['name']}</b></td><td>{r['length']}</td>"
                     f"<td>{r['plasticity']}</td>"
                     f"<td>{r['continuum_class']}</td>"
                     f"<td>{r['foldswitch']} ({r['foldswitch_pct']}%)</td>"
                     f"<td>{r['n_hotspots']}</td></tr>")
        h.append("</table>")
        display(HTML("".join(h)))

def on_dl_csv(_):
    rows = _summary_rows()
    if not rows:
        return
    path = os.path.join(WORK, "proteus_summary.csv")
    with open(path, "w", newline="") as fh:
        wtr = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); wtr.writeheader()
        wtr.writerows(rows)
    try:
        from google.colab import files; files.download(path)
    except Exception:
        print("saved:", path)

def on_dl_json(_):
    out = {nm: {"length": s["length"], "sequence": s.get("seq", ""),
                "l2_delta_max": s["features"]["l2_delta_max"],
                "length_corrected": s["resid"],
                "profile": s["profile"].tolist() if s.get("profile") is not None else None}
           for nm, s in SAMPLES.items() if s["status"] == "done"}
    path = os.path.join(WORK, "proteus_results.json")
    with open(path, "w") as fh:
        json.dump(out, fh, indent=2)
    try:
        from google.colab import files; files.download(path)
    except Exception:
        print("saved:", path)

def on_landscape(_):
    with summary_out:
        clear_output()
        rows = _summary_rows()
        if not rows:
            display(HTML("<div class='pst-sub'>No completed samples yet.</div>")); return
        plot_landscape([{"name": r["name"], "length": r["length"],
                         "resid": r["length_corrected"],
                         "continuum_class": r["continuum_class"]} for r in rows])

def on_overlay(_):
    with summary_out:
        clear_output()
        items = [(nm, SAMPLES[nm]["profile"]) for nm in overlay_sel.value
                 if SAMPLES.get(nm, {}).get("status") == "done"]
        if len(items) < 1:
            display(HTML("<div class='pst-sub'>Select one or more samples in the list."
                         "</div>")); return
        plot_overlay(items)

summary_btn.on_click(on_summary); dl_csv.on_click(on_dl_csv); dl_json.on_click(on_dl_json)
landscape_btn.on_click(on_landscape); overlay_btn.on_click(on_overlay)


# ---------------- layout ----------------
def _card(title, sub, *widgets):
    head = W.HTML(f"<div class='pst-h'>{title}</div><div class='pst-sub'>{sub}</div>")
    box = W.VBox([head, *widgets])
    box.add_class("pst-card")
    return box

panel_add = _card("1 · Add proteins",
                  "Paste FASTA or a bare sequence, or upload .fasta / pre-computed .npz.",
                  seq_box, W.HBox([add_btn, clear_btn, upload]), queue_out)
panel_run = _card("2 · Run PROTEUS",
                  f"Device detected: <b>{DEVICE.upper()}</b>. First run loads ESM-2 3B (fp16).",
                  W.HBox([nconf, run_btn]), run_out)
panel_res = _card("3 · Explore a sample",
                  "Per-residue profile, continuum, fold-switch gauge, 3D map. "
                  "Optionally upload a .pdb/.cif for the 3D view.",
                  W.HBox([result_sel, show3d, show_btn]),
                  W.HBox([struct_up]), result_out)
panel_cmp = _card("4 · Compare across samples & export",
                  "Rank samples, map the plasticity landscape, overlay profiles, and export.",
                  W.HBox([summary_btn, landscape_btn, dl_csv, dl_json]),
                  W.HBox([overlay_sel, overlay_btn]),
                  summary_out)

banner = W.HTML(
    "<div class='proteus-banner'>"
    "<div style='font-size:22px;font-weight:800'>🧬 PROTEUS Studio</div>"
    "<div style='font-size:13px;opacity:.92'>Zero-shot conformational-plasticity "
    "analysis — queue proteins, run, and explore. Free-tier ready.</div></div>")
studio = W.VBox([banner, panel_add, panel_run, panel_res, panel_cmp])
studio.add_class("proteus-studio")
display(studio)
render_queue()